In [1]:
import pandas as pd
import muon as mu
import sys
import os



# Change path to wherever you have repo locally
sys.path.append('/oak/stanford/groups/engreitz/Users/ymo/Tools/cNMF_benchmarking/cNMF_benchmarking_pipeline')


from Plotting.src import rename_adata_gene_dictionary

from Evaluation.src import (
   compile_Program_loading_score_sheet_long, compile_Program_loading_score_sheet_flat, Compile_GO_sheet, \
    Compile_Geneset_sheet, Compile_Trait_sheet, Compile_Perturbation_sheet, Compile_Association_sheet, Compile_Explained_variance, \
    Compile_Target_Summary_sheet, Compile_Summary_sheet, load_simple_sheets
)

from Evaluation.src import (
    check_evaluation_pipeline_format,
    check_gene_names,
    _validate_against_reference_gtf
)


/home/users/ymo/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# respurse path
reference_gtf_path="/oak/stanford/groups/engreitz/Users/opushkar/genome/IGVFFI9573KOZR.gtf.gz"

# IO path
out_dir = '/oak/stanford/groups/engreitz/Users/ymo/cc-perturb-seq/Results'
run_name = '111025_D0_IGVF_10iter_torch_halsvar_batch_e7_v100s_test'

# data path
mdata_path = '/oak/stanford/groups/engreitz/Users/ymo/cc-perturb-seq/Results/111025_D0_IGVF_10iter_torch_halsvar_batch_e7_v100s_test/adata/cNMF_30_0_4.h5mu'

# keys 
prog_key = 'cNMF'
data_key = 'rna'
categorical_key = 'batch'
guide_targets_key = "guide_targets"
num_gene = 300
components = [30, 50, 60, 80, 100, 200]
sel_threshs = [0.4, 0.8, 2.0]
Sample = ['1', '2', '3']

In [6]:
'''# rename genes if needed 
file_to_dictionary = "/oak/stanford/groups/engreitz/Users/ymo/Tools/cNMF_benchmarking/cNMF_benchmarking_pipeline/Evaluation/Resources/weissman_guides_with_coordinates.tsv"
result = rename_adata_gene_dictionary(mdata['rna'] ,dictionary_file_path=file_to_dictionary)
mdata.mod['rna'] = result'''

In [3]:
# reformat if needed 
mdata_guide_path = "/oak/stanford/groups/engreitz/Users/ymo/cc-perturb-seq/Data/IGVF_D0_example.h5mu"
mdata_guide = mu.read(mdata_guide_path)

# helper method 
def _assign_guide(mdata, mdata_guide):
        mdata['rna'].var_names = mdata['rna'].var['symbol']
        mdata['cNMF'].uns['guide_names'] = mdata_guide['guide'].var['guide_id']
        mdata['cNMF'].uns['guide_targets'] = mdata_guide['guide'].var['intended_target_name']
        mdata['cNMF'].obsm['guide_assignment'] = mdata_guide['guide'].layers['guide_assignment'].toarray()


In [ ]:
for sel_thresh in sel_threshs:
    for k in components:  

        output_folder = f"{out_dir}/{run_name}/Eval/{k}_{str(sel_thresh).replace('.','_')}"

        os.makedirs(output_folder, exist_ok=True)

        # Load mdata
        mdata = mu.read('{out_dir}/{run_name}/adata/cNMF_{k}_{sel_thresh}.h5mu'.format(out_dir = out_dir,
                                                                                run_name =run_name,
                                                                                k=k,
                                                                                sel_thresh = str(sel_thresh).replace('.','_')))                                                                      
        # assign guide
        _assign_guide(mdata, mdata_guide)   

        # checks for correct mdata format
        if not check_evaluation_pipeline_format(mdata,prog_key=prog_key):
            raise ValueError("mdata format is incorrect")


        valid = check_gene_names(mdata,prog_key=prog_key, data_key=data_key,categorical_key=categorical_key,reference_gtf_path=reference_gtf_path)
        if not valid["is_valid"]:
            raise ValueError("mdata gene naming is incorrect")

        # load simple sheets
        df_Program_loading_long,df_Program_loading_flat, df_GO, df_Geneset, \
        df_Trait, df_Perturbation, df_Association, df_Explained_Variance = load_simple_sheets( 
            mdata, out_dir, run_name, k, sel_thresh, num_gene = num_gene,  Sample = Sample)

        # load target summary
        Perturbation_path_base = f'{out_dir}/{run_name}/Eval/{k}_{sel_thresh}/{k}_perturbation_association_results'
        df_Target_Summary = Compile_Target_Summary_sheet(mdata, Perturbation_path_base, Sample = Sample, categorical_key = categorical_key, 
        prog_key = prog_key, data_key = data_key, guide_targets_key = guide_targets_key)
        
        # load summary    
        df_Summary = Compile_Summary_sheet(mdata, df_GO, df_Geneset, df_Perturbation, df_Program_loading_flat, df_Explained_Variance, Sample = Sample,
        categorical_key = categorical_key)  


        # save to excel sheets 
        with pd.ExcelWriter(f'{output_folder}/cNMF_{k}_{str(sel_thresh).replace(".", "_")}.xlsx') as writer:
            df_Summary.to_excel(writer, sheet_name='Summary', index=True)
            df_Program_loading_long.to_excel(writer, sheet_name='Program Loadings', index=True)
            df_Target_Summary.to_excel(writer, sheet_name='Targets Summary', index=True)
            df_Association.to_excel(writer, sheet_name='Sample Association', index=True)
            df_Perturbation.to_excel(writer, sheet_name='Perturbation Association', index=True)
            df_Trait.to_excel(writer, sheet_name='Trait Enrichment', index=True)
            df_GO.to_excel(writer, sheet_name='GO Term Enrichment', index=True)
            df_Geneset.to_excel(writer, sheet_name='Geneset Enrichment', index=True)
